In [81]:
from pathlib import Path
import re

import pandas as pd

DATA_PATH = "data"
RE_FILE_YEAR = re.compile(r"resultat-ansokningsomgang-(20\d{2})\.xlsx$")

current_dir = globals()["_dh"][0]  # CWD for jupyter
path_data = Path(current_dir) / DATA_PATH

In [82]:
# EXTRACT

df = pd.read_excel(path_data / "kommunlankod_2025.xls", sheet_name=0, skiprows=5, names=["Kod", "Namn"])


# TRANSFORM

df["is_county"] = df["Kod"].between(1, 25)  # Länskod

df_county = df.loc[df["is_county"], ["Kod", "Namn"]].rename(columns={"Namn": "Län"})

df["Län"] = df.loc[df["is_county"], "Namn"]
df["Län"] = df["Län"].ffill()

df = df.loc[~df["is_county"], ["Kod", "Län", "Namn"]].reset_index(drop=True)

df = pd.concat([df_county, df])

df = df.rename(columns={"Namn": "Kommun", "Län": "Län, namn"})

df["Län"] = df["Län, namn"].str.replace(r"(s län| län)$", "", regex=True)

cols_order = [
    "Kommun",
    "Län",
    "Län, namn",
    "Kod",
]

df = df[cols_order].reset_index(drop=True)


# LOAD

df.to_csv(path_data / "kommunlankod-2025.csv", index=False)


# RESULT

df.head(50)

,Kommun,Län,"Län, namn",Kod
0,NaN,Stockholm,Stockholms län,1
1,NaN,Uppsala,Uppsala län,3
2,NaN,Södermanland,Södermanlands län,4
3,NaN,Östergötland,Östergötlands län,5
4,NaN,Jönköping,Jönköpings län,6
5,NaN,Kronoberg,Kronobergs län,7
6,NaN,Kalmar,Kalmar län,8
7,NaN,Gotland,Gotlands län,9
8,NaN,Blekinge,Blekinge län,10
9,NaN,Skåne,Skåne län,12


In [83]:
df_csv = pd.read_csv(path_data / "kommunlankod-2025.csv").set_index("Kod")

In [84]:
df_lan = df_csv[df_csv["Kommun"].isna()].copy().drop(columns="Kommun")
mapping_lan = df_lan.set_index("Län")["Län, namn"].to_dict()

mapping_lan


{'Stockholm': 'Stockholms län',
 'Uppsala': 'Uppsala län',
 'Södermanland': 'Södermanlands län',
 'Östergötland': 'Östergötlands län',
 'Jönköping': 'Jönköpings län',
 'Kronoberg': 'Kronobergs län',
 'Kalmar': 'Kalmar län',
 'Gotland': 'Gotlands län',
 'Blekinge': 'Blekinge län',
 'Skåne': 'Skåne län',
 'Halland': 'Hallands län',
 'Västra Götaland': 'Västra Götalands län',
 'Värmland': 'Värmlands län',
 'Örebro': 'Örebro län',
 'Västmanland': 'Västmanlands län',
 'Dalarna': 'Dalarnas län',
 'Gävleborg': 'Gävleborgs län',
 'Västernorrland': 'Västernorrlands län',
 'Jämtland': 'Jämtlands län',
 'Västerbotten': 'Västerbottens län',
 'Norrbotten': 'Norrbottens län'}

In [85]:
df_lan_kommun = df_csv[~df_csv["Kommun"].isna()].copy()
mapping_kommun_lan = df_lan_kommun.set_index("Kommun")["Län"].to_dict()

mapping_kommun_lan

{'Upplands Väsby': 'Stockholm',
 'Vallentuna': 'Stockholm',
 'Österåker': 'Stockholm',
 'Värmdö': 'Stockholm',
 'Järfälla': 'Stockholm',
 'Ekerö': 'Stockholm',
 'Huddinge': 'Stockholm',
 'Botkyrka': 'Stockholm',
 'Salem': 'Stockholm',
 'Haninge': 'Stockholm',
 'Tyresö': 'Stockholm',
 'Upplands-Bro': 'Stockholm',
 'Nykvarn': 'Stockholm',
 'Täby': 'Stockholm',
 'Danderyd': 'Stockholm',
 'Sollentuna': 'Stockholm',
 'Stockholm': 'Stockholm',
 'Södertälje': 'Stockholm',
 'Nacka': 'Stockholm',
 'Sundbyberg': 'Stockholm',
 'Solna': 'Stockholm',
 'Lidingö': 'Stockholm',
 'Vaxholm': 'Stockholm',
 'Norrtälje': 'Stockholm',
 'Sigtuna': 'Stockholm',
 'Nynäshamn': 'Stockholm',
 'Håbo': 'Uppsala',
 'Älvkarleby': 'Uppsala',
 'Knivsta': 'Uppsala',
 'Heby': 'Uppsala',
 'Tierp': 'Uppsala',
 'Uppsala': 'Uppsala',
 'Enköping': 'Uppsala',
 'Östhammar': 'Uppsala',
 'Vingåker': 'Södermanland',
 'Gnesta': 'Södermanland',
 'Nyköping': 'Södermanland',
 'Oxelösund': 'Södermanland',
 'Flen': 'Södermanland',
 'Kat